**Instalar funções necessárias**

In [1]:
!pip install unidecode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 4.7 MB/s eta 0:00:00


**Bibliotecas para Web Scraping, manipulação de texto e dados**

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
from unidecode import unidecode

1. Carregamento e preparação inicial dos dados

In [ ]:
# Lê o CSV contendo a lista de Pokémon
df_csv = pd.read_csv('pokedex.csv')
# Cria uma nova coluna com o nome do Pokémon normalizado (sem acentos, em minúsculas)
df_csv['name_clean'] = df_csv['name'].apply(lambda x: unidecode(str(x)).lower().strip())

In [ ]:
# Extrai apenas o primeiro tipo (caso haja mais de um) e limpa a string
df_csv['type'] = df_csv['type'].astype(str).apply(lambda x: re.sub(r'[\{\}\s]', '', x.split(',')[0].lower()))
# Mostra as primeiras 10 entradas do CSV para verificar a leitura
df_preview = df_csv.head(10).copy()
print("Preview do CSV original:")
print(df_preview[['name', 'type']])

Preview do CSV original:
         name   type
0   bulbasaur  grass
1     ivysaur  grass
2    venusaur  grass
3  charmander   fire
4  charmeleon   fire
5   charizard   fire
6    squirtle  water
7   wartortle  water
8   blastoise  water
9    caterpie    bug


2. Função de Web Scraping para extrair a seção "Biology" da Bulbapedia

In [ ]:
# 2. Função para buscar a secção "Biology" de cada Pokémon
def get_bulbapedia_biology(pokemon_name):
    # Formata o nome para a URL
    url_name = pokemon_name.replace(' ', '_')
    url = f"https://bulbapedia.bulbagarden.net/wiki/{url_name}_(Pok%C3%A9mon)"
    headers = {'User-Agent': 'Mozilla/5.0'}
    try:
        resp = requests.get(url, headers=headers, timeout=10)
        if resp.status_code != 200:
            print(f"Failed to fetch {pokemon_name}: {resp.status_code}")
            return ""
        soup = BeautifulSoup(resp.content, "html.parser")
        # Procura pela seção Biology
        header = soup.find('span', {'id': 'Biology'})
        if not header:
            print(f"No Biology section for {pokemon_name}")
            return ""
        # Navega pelos elementos após o título "Biology" até encontrar outro h2
        section = header.find_parent('h2')
        section_text = []
        next_element = section.find_next_sibling()
        while next_element and next_element.name != 'h2':
            if next_element.name in ('p', 'ul', 'li'):
                section_text.append(next_element.get_text(separator=" ", strip=True))
            next_element = next_element.find_next_sibling()
        return '\n'.join(section_text)
    except Exception as e:
        print(f"Error fetching {pokemon_name}: {e}")
        return ""

# Webscraping para todos os Pokémon do CSV
scraped_data = []
for idx, row in df_csv.iterrows():
    name = row['name']
    name_clean = unidecode(str(name)).lower().strip()
# Substitui alguns caracteres antes de passar para a URL
    bio = get_bulbapedia_biology(name.replace('-', ' ').replace('.', '').replace("’", "'"))
    scraped_data.append({
        'name': name,
        'bio': bio
    })
    print(f"{idx+1}/{len(df_csv)} - {name} done.")
    time.sleep(1.5)  # Respeitar o site
# Converte os dados obtidos para um DataFrame
df_bulba = pd.DataFrame(scraped_data)
df_bulba['name_clean'] = df_bulba['name'].apply(lambda x: unidecode(str(x)).lower().strip())

1/1025 - bulbasaur done.
2/1025 - ivysaur done.
3/1025 - venusaur done.
4/1025 - charmander done.
5/1025 - charmeleon done.
6/1025 - charizard done.
7/1025 - squirtle done.
8/1025 - wartortle done.
9/1025 - blastoise done.
10/1025 - caterpie done.
11/1025 - metapod done.
12/1025 - butterfree done.
13/1025 - weedle done.
14/1025 - kakuna done.
15/1025 - beedrill done.
16/1025 - pidgey done.
17/1025 - pidgeotto done.
18/1025 - pidgeot done.
19/1025 - rattata done.
20/1025 - raticate done.
21/1025 - spearow done.
22/1025 - fearow done.
23/1025 - ekans done.
24/1025 - arbok done.
25/1025 - pikachu done.
26/1025 - raichu done.
27/1025 - sandshrew done.
28/1025 - sandslash done.
Failed to fetch nidoran f: 404
29/1025 - nidoran-f done.
30/1025 - nidorina done.
31/1025 - nidoqueen done.
Failed to fetch nidoran m: 404
32/1025 - nidoran-m done.
33/1025 - nidorino done.
34/1025 - nidoking done.
35/1025 - clefairy done.
36/1025 - clefable done.
37/1025 - vulpix done.
38/1025 - ninetales done.
39/1

In [ ]:
print("Head do DataFrame do scraping:")
print(df_bulba.head(10))


Head do DataFrame do scraping:


NameError: name 'df_bulba' is not defined

Merge do df do scraping com o CSV original para garantir alinhamento


In [ ]:
# Merge com o CSV original para garantir alinhamento
df_merged = pd.merge(df_bulba, df_csv, on='name_clean', how='inner', suffixes=('_bulba', '_csv'))
df_merged = df_merged[['name_bulba', 'bio', 'type']]
df_merged = df_merged.rename(columns={'name_bulba': 'name', 'bio': 'description'})

print("\nHead do DataFrame final (merge):")
print(df_merged.head())


Head do DataFrame final (merge):
         name                                        description   type
0   bulbasaur  Bulbasaur is a small, quadrupedal amphibian Po...  grass
1     ivysaur  Ivysaur is a quadrupedal amphibian Pokémon tha...  grass
2    venusaur  Venusaur is a squat, quadrupedal amphibian Pok...  grass
3  charmander  Charmander is a bipedal, reptilian Pokémon wit...   fire
4  charmeleon  Charmeleon is a bipedal, reptilian Pokémon . I...   fire


In [ ]:
# Salvar para CSV
df_merged.to_csv('merged_pokemon_biology.csv', index=False)

Começa a parte de Processamento de Texto e Classificação

In [3]:
# Importação das bibliotecas de NLP e ML
import pandas as pd
import numpy as np
import re
import nltk
from sklearn.decomposition import NMF
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from unidecode import unidecode
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

In [ ]:
# Pré-processamento com NLTK
nltk.download('stopwords')
nltk.download('wordnet')
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


In [ ]:
# Carregar e preparar dados
df_final = pd.read_csv('merged_pokemon_biology.csv')
# Mantém apenas os 7 tipos mais comuns para menor diversidade
top_types = df_final['type'].value_counts().nlargest(7).index.tolist()
df_final = df_final[df_final['type'].isin(top_types)].reset_index(drop=True)


In [ ]:
# Balancear o dataset com undersampling
from sklearn.utils import resample
min_count = df_final['type'].value_counts().min()
dfs = []
for t in df_final['type'].unique():
    dfs.append(df_final[df_final['type'] == t].sample(min_count, random_state=42))
df_balanced = pd.concat(dfs).reset_index(drop=True)

In [ ]:
print("Distribuição após balanceamento:")
print(df_balanced['type'].value_counts())


Distribuição após balanceamento:
type
grass       59
fire        59
water       59
bug         59
normal      59
electric    59
psychic     59
Name: count, dtype: int64


In [ ]:
# Função de limpeza de texto
def clean_text(text):
    text = str(text).lower()
    text = unidecode(text)
    # Remove palavras comuns e ruído textual
    text = re.sub(r'\bpokemon\b', '', text)
    text = re.sub(r'nan', '', text)
    text = re.sub(r'game', '', text)
    text = re.sub(r'body', '', text)
    text = re.sub(r'one', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = [lemmatizer.lemmatize(word) for word in text.split() if word not in stop_words]
    return ' '.join(tokens)
# Aplica a limpeza no texto
df_balanced['desc_clean'] = df_balanced['description'].astype(str).apply(clean_text)

In [ ]:
print(df_balanced.head())

        name                                        description   type  \
0     cacnea  Cacnea is a green, bipedal Pokémon with a roun...  grass   
1    dartrix  Dartrix is an avian Pokémon that resembles an ...  grass   
2  quilladin  Quilladin is a bipedal, mammalian Pokémon with...  grass   
3    servine  Servine is a slim bipedal Pokémon that is prim...  grass   
4  abomasnow  Abomasnow is a large, bipedal Pokémon covered ...  grass   

                                          desc_clean  
0  cacnea green bipedal round cactuslike striatio...  
1  dartrix avian resembles owl plumage including ...  
2  quilladin bipedal mammalian plantlike feature ...  
3  servine slim bipedal primarily green cream und...  
4  abomasnow large bipedal covered shaggy white f...  


In [ ]:
# Vetorização TF-IDF
vectorizer_tfidf = TfidfVectorizer(max_df=0.95, min_df=3)
tfidf_matrix = vectorizer_tfidf.fit_transform(df_balanced['desc_clean'])
tfidf_features = vectorizer_tfidf.get_feature_names_out()


In [ ]:
# Modelação de tópicos com NMF
def show_topics(model, feature_names, n_top_words=10):
    for topic_idx, topic in enumerate(model.components_):
        print(f"Tópico {topic_idx+1}: ", " | ".join([feature_names[i] for i in topic.argsort()[:-n_top_words - 1:-1]]))

def nmf_topics(tfidf_matrix, feature_names, n_topics=5, n_top_words=10):
    nmf = NMF(n_components=n_topics, random_state=42)
    W = nmf.fit_transform(tfidf_matrix)
    print("\n=== Tópicos NMF ===")
    show_topics(nmf, feature_names, n_top_words)
    return nmf, W

nmf_model, nmf_W = nmf_topics(tfidf_matrix, tfidf_features, n_topics=5, n_top_words=10)


=== Tópicos NMF ===
Tópico 1:  feather | wing | black | beak | avian | marking | head | crest | two | talon
Tópico 2:  fur | flame | ear | sleep | tail | tuft | brown | paw | evolves | nose
Tópico 3:  leaf | shell | green | evolves | head | tree | forest | evolution | mouth | vine
Tópico 4:  fin | water | blue | tail | prey | sea | scale | pectoral | dorsal | light
Tópico 5:  power | form | electricity | yellow | blue | head | two | capable | known | white


/usr/local/lib/python3.11/dist-packages/sklearn/decomposition/_nmf.py:1742: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(


In [ ]:
# Soma dos valores TF-IDF para cada termo (coluna)
soma_tfidf = np.asarray(tfidf_matrix.sum(axis=0)).ravel()
termos = np.array(vectorizer_tfidf.get_feature_names_out())

# Ordenar termos pelo valor total de TF-IDF (do maior para o menor)
indices = soma_tfidf.argsort()[::-1]
top_termos = termos[indices]
top_somas = soma_tfidf[indices]

# Mostrar os 10 termos mais relevantes
print("Top 10 termos pelo total de TF-IDF:")
for termo, soma in zip(top_termos[:10], top_somas[:10]):
    print(f"{termo}: {soma:.2f}")


Top 10 termos pelo total de TF-IDF:
black: 15.01
head: 14.97
yellow: 14.34
two: 13.57
tail: 13.20
evolves: 12.34
fur: 12.23
eye: 11.42
red: 11.30
blue: 11.27


In [ ]:
# 4. Features e target
X = tfidf_matrix
y = df_balanced['type']

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

def avalia_modelos(X, y, nome_repr):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

    # Regressão Logística
    clf = LogisticRegression(max_iter=1000, multi_class='multinomial')
    clf.fit(X_train, y_train)
    y_pred_lr = clf.predict(X_test)
    print(f"\n=== Logistic Regression ({nome_repr}) ===")
    print("Accuracy:", accuracy_score(y_test, y_pred_lr))
    print(classification_report(y_test, y_pred_lr))

    # Random Forest
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X_train, y_train)
    y_pred_rf = rf.predict(X_test)
    print(f"\n=== Random Forest ({nome_repr}) ===")
    print("Accuracy:", accuracy_score(y_test, y_pred_rf))
    print(classification_report(y_test, y_pred_rf))

    # Naive Bayes
    nb = MultinomialNB()
    nb.fit(X_train, y_train)
    y_pred_nb = nb.predict(X_test)
    print(f"\n=== Naive Bayes ({nome_repr}) ===")
    print("Accuracy:", accuracy_score(y_test, y_pred_nb))
    print(classification_report(y_test, y_pred_nb))


In [ ]:
# Chamada da função para treinar e avaliar os modelos
avalia_modelos(X, y, "NMF Topics")



/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(



=== Logistic Regression (NMF Topics) ===
Accuracy: 0.8192771084337349
              precision    recall  f1-score   support

         bug       1.00      0.82      0.90        11
    electric       1.00      0.83      0.91        12
        fire       1.00      0.83      0.91        12
       grass       0.92      0.92      0.92        12
      normal       0.75      0.50      0.60        12
     psychic       0.52      0.92      0.67        12
       water       0.85      0.92      0.88        12

    accuracy                           0.82        83
   macro avg       0.86      0.82      0.83        83
weighted avg       0.86      0.82      0.83        83


=== Random Forest (NMF Topics) ===
Accuracy: 0.7590361445783133
              precision    recall  f1-score   support

         bug       0.92      1.00      0.96        11
    electric       1.00      0.75      0.86        12
        fire       1.00      0.75      0.86        12
       grass       0.91      0.83      0.87       

In [ ]:
# Logistic Regression
clf = LogisticRegression(max_iter=1000, multi_class='multinomial')
clf.fit(X_train, y_train)
y_pred_lr = clf.predict(X_test)
print("\n=== Logistic Regression ===")
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(



=== Logistic Regression ===
Accuracy: 0.776
              precision    recall  f1-score   support

         bug       1.00      0.94      0.97        16
    electric       1.00      0.67      0.80        12
        fire       1.00      0.62      0.76        13
       grass       0.94      0.81      0.87        21
      normal       0.58      0.75      0.65        24
     psychic       1.00      0.50      0.67        12
       water       0.64      0.93      0.76        27

    accuracy                           0.78       125
   macro avg       0.88      0.74      0.78       125
weighted avg       0.83      0.78      0.78       125



In [ ]:
# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print("\n=== Random Forest ===")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))


=== Random Forest ===
Accuracy: 0.688
              precision    recall  f1-score   support

         bug       0.75      0.75      0.75        16
    electric       0.89      0.67      0.76        12
        fire       1.00      0.62      0.76        13
       grass       0.71      0.81      0.76        21
      normal       0.56      0.62      0.59        24
     psychic       0.83      0.42      0.56        12
       water       0.60      0.78      0.68        27

    accuracy                           0.69       125
   macro avg       0.76      0.67      0.69       125
weighted avg       0.72      0.69      0.69       125



In [ ]:
# Naive Bayes
nb = MultinomialNB()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)
print("\n=== Naive Bayes ===")
print("Accuracy:", accuracy_score(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb))


=== Naive Bayes ===
Accuracy: 0.592
              precision    recall  f1-score   support

         bug       1.00      0.56      0.72        16
    electric       0.00      0.00      0.00        12
        fire       1.00      0.15      0.27        13
       grass       1.00      0.76      0.86        21
      normal       0.42      0.83      0.56        24
     psychic       1.00      0.08      0.15        12
       water       0.53      0.96      0.68        27

    accuracy                           0.59       125
   macro avg       0.71      0.48      0.46       125
weighted avg       0.69      0.59      0.53       125



/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
